# Pharmasense — TFT: Full Drug Demand Forecasting
**Temporal Fusion Transformer — ALL conditions, CSV-based, no DB required**

| Condition | Type | Source |
|---|---|---|
| malaria | Communicable | NVBDCP 2010-2024 (WHO-anchored) |
| dengue, cholera, acute_diarrhoeal_disease | Communicable | EpiClim IDSP |
| hypertension, type2_diabetes, tuberculosis, anaemia | NCD chronic | NFHS-5 MBQ |
| viral_fever_urti, musculoskeletal_pain, etc. | Symptomatic | NSSO 75th Round MBQ |

## Before running
1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run All** (no secrets or uploads needed)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/02_train_tft.ipynb)

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(f'GPU: {r.stdout.strip()}')
else:
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> T4 GPU')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pytorch-forecasting==1.0.0', 'pytorch-lightning==2.2.4',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn'], check=True)
print('Dependencies installed.')

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
GITHUB_RAW = 'https://raw.githubusercontent.com/abhi-dev99/agtechathon/main/ml/data'

MAX_ENCODER_LEN    = 24
MAX_PREDICTION_LEN = 4
MIN_OBS_PER_GROUP  = 30
BATCH_SIZE         = 128
MAX_EPOCHS         = 60
LEARNING_RATE      = 3e-3
HIDDEN_SIZE        = 48
ATTENTION_HEADS    = 4
DROPOUT            = 0.1
HIDDEN_CONT_SIZE   = 20

DRIVE_OUTPUT = '/content/drive/MyDrive/pharmasense_models/'

import torch
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Drive ready: {DRIVE_OUTPUT}')

In [ ]:
# ── Download training CSVs from GitHub ──────────────────────────────────────
import subprocess
FILES = ['malaria_weekly.csv', 'epiclim_clean.csv', 'ncd_demand_weekly.csv']
for f in FILES:
    url = f'{GITHUB_RAW}/{f}'
    subprocess.run(['wget', '-q', '-O', f'/content/{f}', url], check=True)
    size = os.path.getsize(f'/content/{f}') / 1e6
    print(f'  {f}: {size:.1f} MB')
print('All data files downloaded.')

In [ ]:
# ── Load and unify all data sources ─────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ── Source 1: Malaria (NVBDCP weekly, 2010-2024) ─────────────────────────────
mal = pd.read_csv('/content/malaria_weekly.csv', parse_dates=['week_start_date'])
mal = mal[['week_start_date','state','district','disease','cases']].rename(
    columns={'disease':'condition_type','cases':'raw_value'})
mal['data_family'] = 'communicable'
print(f'Malaria rows: {len(mal):,} | districts: {mal.district.nunique()}')

# ── Source 2: Epiclim (dengue, cholera, diarrhoea) ───────────────────────────
epi = pd.read_csv('/content/epiclim_clean.csv', parse_dates=['week_of_outbreak'],
                  dayfirst=False)
epi = epi.rename(columns={'week_of_outbreak':'week_start_date',
                           'state_ut':'state','Disease':'condition_type',
                           'Cases':'raw_value'})
comm_diseases = ['Dengue','Cholera','Acute Diarrhoeal Disease','dengue','cholera',
                 'acute_diarrhoeal_disease']
epi = epi[epi['condition_type'].isin(comm_diseases)].copy()
epi['condition_type'] = epi['condition_type'].str.lower().str.replace(' ','_')
epi['data_family'] = 'communicable'
epi = epi[['week_start_date','state','district','condition_type','raw_value','data_family']]
print(f'Epiclim rows: {len(epi):,} | conditions: {epi.condition_type.unique()}')

# ── Source 3: NCD / symptomatic demand ───────────────────────────────────────
ncd = pd.read_csv('/content/ncd_demand_weekly.csv', parse_dates=['week_start_date'])
ncd = ncd.rename(columns={'condition':'condition_type','patients_est':'raw_value'})
ncd['data_family'] = 'ncd'
ncd = ncd[['week_start_date','state','district','condition_type','raw_value','data_family']]
print(f'NCD rows: {len(ncd):,} | conditions: {ncd.condition_type.nunique()}')

# ── Combine ───────────────────────────────────────────────────────────────────
df = pd.concat([mal, epi, ncd], ignore_index=True)
df['raw_value'] = pd.to_numeric(df['raw_value'], errors='coerce').fillna(0).clip(lower=0)
df['log_value'] = np.log1p(df['raw_value'])
df['month'] = df['week_start_date'].dt.month.astype(str)

# Encode categorical features
le_district  = LabelEncoder().fit(df['district'])
le_state     = LabelEncoder().fit(df['state'])
le_condition = LabelEncoder().fit(df['condition_type'])
le_family    = LabelEncoder().fit(df['data_family'])

df['district_enc']  = le_district.transform(df['district'])
df['state_enc']     = le_state.transform(df['state'])
df['condition_enc'] = le_condition.transform(df['condition_type'])
df['family_enc']    = le_family.transform(df['data_family'])

# Integer time index per group
df = df.sort_values(['district','condition_type','week_start_date'])
df['group_id'] = df['district'] + '__' + df['condition_type']
df['time_idx'] = df.groupby('group_id').cumcount()

# Filter groups with enough observations
group_counts = df.groupby('group_id')['time_idx'].max()
valid_groups = group_counts[group_counts >= MIN_OBS_PER_GROUP].index
df = df[df['group_id'].isin(valid_groups)].reset_index(drop=True)

print(f'\nCombined dataset: {len(df):,} records')
print(f'Condition types:  {df["condition_type"].nunique()}')
print(f'Districts:        {df["district"].nunique()}')
print(f'Valid groups:     {df["group_id"].nunique()}')
print(f'Date range:       {df["week_start_date"].min().date()} to {df["week_start_date"].max().date()}')
print()
print(df.groupby('condition_type')['raw_value'].agg(['count','mean']).round(1))

In [ ]:
# ── Build TimeSeriesDataSet ──────────────────────────────────────────────────
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

cutoff = df['time_idx'].max() - MAX_PREDICTION_LEN

training = TimeSeriesDataSet(
    df[df['time_idx'] <= cutoff],
    time_idx='time_idx',
    target='log_value',
    group_ids=['group_id'],
    min_encoder_length=MAX_ENCODER_LEN // 2,
    max_encoder_length=MAX_ENCODER_LEN,
    min_prediction_length=1,
    max_prediction_length=MAX_PREDICTION_LEN,
    static_categoricals=['district_enc','state_enc','condition_enc','family_enc'],
    time_varying_known_reals=['time_idx'],
    time_varying_unknown_reals=['log_value'],
    time_varying_known_categoricals=['month'],
    target_normalizer=GroupNormalizer(groups=['group_id'], transformation='softplus'),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)

train_dl = training.to_dataloader(train=True,  batch_size=BATCH_SIZE, num_workers=2)
val_dl   = validation.to_dataloader(train=False, batch_size=BATCH_SIZE*2, num_workers=2)

print(f'Training samples:   {len(training):,}')
print(f'Validation samples: {len(validation):,}')

In [ ]:
# ── Build TFT model ──────────────────────────────────────────────────────────
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=LEARNING_RATE,
    hidden_size=HIDDEN_SIZE,
    attention_head_size=ATTENTION_HEADS,
    dropout=DROPOUT,
    hidden_continuous_size=HIDDEN_CONT_SIZE,
    loss=QuantileLoss(quantiles=[0.1, 0.25, 0.5, 0.75, 0.9]),
    log_interval=5,
    reduce_on_plateau_patience=4,
)
print(f'TFT parameters: {tft.size()/1e3:.1f}k')

callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, mode='min', verbose=True),
    ModelCheckpoint(monitor='val_loss', filename='tft-best-{epoch:02d}-{val_loss:.4f}',
                    save_top_k=1, mode='min'),
    LearningRateMonitor(),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='gpu', devices=1,
    enable_model_summary=True,
    callbacks=callbacks,
    gradient_clip_val=0.1,
    log_every_n_steps=5,
)

In [ ]:
# ── Train ────────────────────────────────────────────────────────────────────
trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)
best_tft = TemporalFusionTransformer.load_from_checkpoint(callbacks[1].best_model_path)
print(f'Best val_loss: {callbacks[1].best_model_score:.6f}')

In [ ]:
# ── Training loss curve ──────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

metrics = trainer.callback_metrics
train_losses = [v.item() for k, v in metrics.items() if 'train_loss' in k.lower()]

# Read from TensorBoard logs
try:
    import glob
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    log_dir = trainer.log_dir or 'lightning_logs'
    event_files = glob.glob(f'{log_dir}/**/events.out.*', recursive=True)
    if event_files:
        ea = EventAccumulator(event_files[0])
        ea.Reload()
        train_loss_steps = ea.Scalars('train_loss_epoch') if 'train_loss_epoch' in ea.Tags()['scalars'] else []
        val_loss_steps   = ea.Scalars('val_loss') if 'val_loss' in ea.Tags()['scalars'] else []
        fig, ax = plt.subplots(figsize=(10, 4))
        if train_loss_steps:
            ax.plot([s.step for s in train_loss_steps], [s.value for s in train_loss_steps],
                    label='Train Loss', color='#2563EB', linewidth=2)
        if val_loss_steps:
            ax.plot([s.step for s in val_loss_steps], [s.value for s in val_loss_steps],
                    label='Val Loss', color='#DC2626', linewidth=2, linestyle='--')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Quantile Loss')
        ax.set_title('TFT Training & Validation Loss')
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(DRIVE_OUTPUT + 'tft_loss_curve.png', dpi=150)
        plt.show()
        print('Loss curve saved.')
except Exception as e:
    print(f'Loss curve via tensorboard skipped: {e}. Check Colab sidebar for built-in plots.')

In [ ]:
# ── Quantitative metrics on validation set ───────────────────────────────────
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

predictions = best_tft.predict(val_dl, return_y=True, trainer_kwargs=dict(accelerator='gpu'))

# Median forecast (q50 = index 2 in [0.1, 0.25, 0.5, 0.75, 0.9])
y_pred_log = predictions.output[:, :, 2].cpu().numpy().flatten()  # q50
y_true_log = predictions.y[0].cpu().numpy().flatten()

# Back-transform from log1p scale
y_pred = np.expm1(np.clip(y_pred_log, 0, None))
y_true = np.expm1(np.clip(y_true_log, 0, None))

mask = y_true > 0  # exclude zero-case periods from MAPE

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
r2   = r2_score(y_true, y_pred)

print('=' * 50)
print('  TFT VALIDATION METRICS (all conditions)')
print('=' * 50)
print(f'  MAE  (cases/week/district): {mae:>10.2f}')
print(f'  RMSE (cases/week/district): {rmse:>10.2f}')
print(f'  MAPE (non-zero only):       {mape:>9.1f}%')
print(f'  R2 score:                   {r2:>10.4f}')
print('=' * 50)

In [ ]:
# ── Forecast plots: predicted vs actual (8 sample series) ────────────────────
import seaborn as sns
sns.set_style('whitegrid')

n = len(validation)
fig, axes = plt.subplots(2, 4, figsize=(22, 9))
for i, ax in enumerate(axes.flat):
    idx = i * max(1, n // 8)
    try:
        best_tft.plot_prediction(predictions.x, predictions.output,
                                 idx=min(idx, n-1), ax=ax, show_future_observed=True)
        group_name = df.iloc[min(idx * 4, len(df)-1)]['group_id'].split('__')
        ax.set_title(f'{group_name[-1]} | {group_name[0][:15]}', fontsize=8)
        ax.set_xlabel(''); ax.set_ylabel('log cases')
    except Exception as e:
        ax.text(0.5, 0.5, str(e), ha='center', va='center', fontsize=7)
plt.suptitle('TFT Forecasts (q10/q50/q90) — All Conditions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'tft_forecasts.png', dpi=150)
plt.show()
print('Forecast plot saved.')

In [ ]:
# ── Residual distribution plot ────────────────────────────────────────────────
residuals = y_pred - y_true
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals, bins=80, color='#3B82F6', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Residual Distribution (pred - actual)')
axes[0].set_xlabel('Residual (cases/week)')
axes[0].set_ylabel('Count')

axes[1].scatter(y_true[:5000], y_pred[:5000], alpha=0.25, s=6, color='#3B82F6')
lim = max(y_true[:5000].max(), y_pred[:5000].max())
axes[1].plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Perfect')
axes[1].set_title(f'Actual vs Predicted (R²={r2:.3f})')
axes[1].set_xlabel('Actual'); axes[1].set_ylabel('Predicted')
axes[1].legend()

plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'tft_residuals.png', dpi=150)
plt.show()
print('Residual plots saved.')

In [ ]:
# ── Variable importance ───────────────────────────────────────────────────────
try:
    raw_pred, x = best_tft.predict(val_dl, mode='raw', return_x=True)
    imp = best_tft.calculate_prediction_actual_by_variable(x, raw_pred)
    best_tft.plot_prediction_actual_by_variable(imp)
    plt.suptitle('TFT Feature Importance — What the model learned to use', fontweight='bold')
    plt.tight_layout()
    plt.savefig(DRIVE_OUTPUT + 'tft_variable_importance.png', dpi=150)
    plt.show()
    print('Variable importance saved.')
except Exception as e:
    print(f'Variable importance skipped: {e}')

In [ ]:
# ── Save artifacts ────────────────────────────────────────────────────────────
import torch, shutil
from datetime import datetime

ckpt_path  = callbacks[1].best_model_path
drive_ckpt = DRIVE_OUTPUT + 'tft_best.ckpt'
shutil.copy2(ckpt_path, drive_ckpt)

artifact = {
    'model_checkpoint_path': drive_ckpt,
    'dataset_parameters': training.get_parameters(),
    'training_config': {
        'all_conditions': le_condition.classes_.tolist(),
        'max_encoder_length': MAX_ENCODER_LEN,
        'max_prediction_length': MAX_PREDICTION_LEN,
    },
    'label_encoders': {'district': le_district, 'state': le_state,
                       'condition': le_condition, 'family': le_family},
    'metrics': {'MAE': float(mae), 'RMSE': float(rmse),
                'MAPE_pct': float(mape), 'R2': float(r2)},
    'best_val_loss': float(callbacks[1].best_model_score),
    'trained_at': datetime.now().isoformat(),
    'n_series': df['group_id'].nunique(),
    'n_records': len(df),
    'data_sources': {'malaria': 'NVBDCP 2010-2024 (weekly)',
                     'epiclim': 'EpiClim IDSP 2009-2022 (weekly)',
                     'ncd': 'NFHS-5 MBQ 2018-2024'},
}
meta_path = DRIVE_OUTPUT + 'tft_latest.pt'
torch.save(artifact, meta_path)

from google.colab import files
files.download(meta_path)
files.download(drive_ckpt)
print(f'Checkpoint: {drive_ckpt}')
print(f'Metadata:   {meta_path}')
print(f'Val loss:   {callbacks[1].best_model_score:.6f}')
print(f'MAPE:       {mape:.1f}%  |  R2: {r2:.4f}')
print('Download both files and save to ml/artifacts/ in your local repo.')